In [4]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path
from urllib.parse import urlparse
from collections import Counter
from bs4 import BeautifulSoup

# =========================
# 1. PATHS
# =========================
JSON_PATH = Path("/Users/timvandegevel/Downloads/profiling-data/results_expanded.json")
HTML_DIR  = Path("/Users/timvandegevel/Downloads/profiling-data/html_front")
CSS_DIR   = Path("/Users/timvandegevel/Downloads/profiling-data/css_front")


# =========================
# 2. LOAD DATA
# =========================
df = pd.read_json(JSON_PATH, lines=True)

# Keep only the two thesis classes
df = df[df["mbfc-credibility-rating"].isin(["high credibility", "medium credibility"])].copy()

# Binary target
df["target"] = df["mbfc-credibility-rating"].map({
    "high credibility": 0,
    "medium credibility": 1
})


# =========================
# 3. MATCH URLS TO FILES
# =========================
html_files = set(os.listdir(HTML_DIR))
css_files = set(os.listdir(CSS_DIR))

def get_filename_from_url(url, extension):
    try:
        domain = urlparse(url).netloc
        return f"{domain}.{extension}"
    except:
        return None

df["html_filename"] = df["url"].apply(lambda x: get_filename_from_url(x, "html"))
df["css_filename"]  = df["url"].apply(lambda x: get_filename_from_url(x, "css"))

df["has_html"] = df["html_filename"].apply(lambda x: x in html_files)
df["has_css"]  = df["css_filename"].apply(lambda x: x in css_files)

# Final thesis sample: keep only rows with BOTH files
df["has_both_files"] = df["has_html"] & df["has_css"]
analysis_df = df[df["has_both_files"]].copy()

# Full paths
analysis_df["html_file_path"] = analysis_df["html_filename"].apply(lambda x: str(HTML_DIR / x))
analysis_df["css_file_path"]  = analysis_df["css_filename"].apply(lambda x: str(CSS_DIR / x))

print("Final sample shape:", analysis_df.shape)
print(analysis_df["mbfc-credibility-rating"].value_counts())

# Optional hard check
assert analysis_df.shape[0] == 3130, f"Expected 3130 rows, got {analysis_df.shape[0]}"
assert analysis_df["target"].value_counts().to_dict() == {0: 2561, 1: 569}, \
    f"Unexpected class counts: {analysis_df['target'].value_counts().to_dict()}"

Final sample shape: (3130, 44)
mbfc-credibility-rating
high credibility      2561
medium credibility     569
Name: count, dtype: int64


In [5]:
# =========================
# 4. HELPER FUNCTIONS
# =========================
def tokenize_css_name(name):
    """
    Split CSS class names into tokens.
    Example:
    'main-nav__item' -> ['main', 'nav', 'item']
    'article_card'   -> ['article', 'card']
    """
    if not isinstance(name, str):
        return []
    
    name = name.strip()
    if not name:
        return []
    
    # Normalize common naming separators
    name = name.replace("__", "-")
    tokens = re.split(r"[-_\s]+", name)
    return [t.lower() for t in tokens if t.strip()]


def shannon_entropy(tokens):
    if not tokens:
        return 0.0
    
    counts = Counter(tokens)
    probs = np.array(list(counts.values()), dtype=float)
    probs = probs / probs.sum()
    return float(-np.sum(probs * np.log2(probs)))


def count_unique_css_variables(css_text):
    """
    Count unique native CSS custom properties, e.g.:
    --main-color: #fff;
    """
    if not isinstance(css_text, str) or not css_text.strip():
        return 0
    
    matches = re.findall(r'(?<![\w-])(--[A-Za-z0-9_-]+)\s*:', css_text)
    return len(set(matches))

In [6]:
# =========================
# 5. FEATURE EXTRACTION
# =========================
def extract_css_features(html_file_path, css_file_path):
    """
    Extract the final CSS feature set for one website/page.
    
    Features:
    - css_class_token_entropy
    - css_reuse_factor
    - structured_naming_ratio
    - inline_style_density
    - css_var_count
    """
    output = {
        "css_class_token_entropy": 0.0,
        "css_reuse_factor": 0.0,
        "structured_naming_ratio": 0.0,
        "inline_style_density": 0.0,
        "css_var_count": 0
    }
    
    try:
        # ----- Read HTML -----
        with open(html_file_path, "r", encoding="utf-8", errors="ignore") as f:
            soup = BeautifulSoup(f.read(), "html.parser")
        
        all_tags = soup.find_all()
        total_tags = len(all_tags)

        class_names = []
        inline_style_count = 0

        for tag in all_tags:
            if tag.has_attr("class"):
                tag_classes = tag.get("class", [])
                class_names.extend([
                    c for c in tag_classes
                    if isinstance(c, str) and c.strip()
                ])
            
            if tag.has_attr("style"):
                inline_style_count += 1

        # 1. css_class_token_entropy
        all_tokens = []
        for class_name in class_names:
            all_tokens.extend(tokenize_css_name(class_name))
        output["css_class_token_entropy"] = shannon_entropy(all_tokens)

        # 2. css_reuse_factor
        unique_classes = len(set(class_names))
        total_class_assignments = len(class_names)
        output["css_reuse_factor"] = (
            total_class_assignments / unique_classes if unique_classes > 0 else 0.0
        )

        # 3. structured_naming_ratio
        structured_count = sum(
            1 for name in class_names
            if ("-" in name) or ("_" in name) or ("__" in name)
        )
        output["structured_naming_ratio"] = (
            structured_count / len(class_names) if len(class_names) > 0 else 0.0
        )

        # 4. inline_style_density
        output["inline_style_density"] = (
            inline_style_count / total_tags if total_tags > 0 else 0.0
        )

        # ----- Read CSS -----
        with open(css_file_path, "r", encoding="utf-8", errors="ignore") as f:
            css_text = f.read()

        # 5. css_var_count
        output["css_var_count"] = count_unique_css_variables(css_text)

    except Exception as e:
        print(f"Error processing {html_file_path}: {e}")

    return output

In [7]:
# =========================
# 6. RUN FEATURE EXTRACTION
# =========================
feature_rows = []

for row in analysis_df.itertuples(index=False):
    feats = extract_css_features(
        html_file_path=row.html_file_path,
        css_file_path=row.css_file_path
    )
    feature_rows.append(feats)

css_feature_df = pd.DataFrame(feature_rows)

# Merge back
result_df = pd.concat(
    [analysis_df.reset_index(drop=True), css_feature_df.reset_index(drop=True)],
    axis=1
)

print(result_df.shape)

(3130, 49)


In [8]:
css_cols = [
    "css_class_token_entropy",
    "css_reuse_factor",
    "structured_naming_ratio",
    "inline_style_density",
    "css_var_count"
]

print("Final dataset shape:")
print(result_df.shape)

print("\nMissing values per feature:")
print(result_df[css_cols].isna().sum().to_string())

print("\nDescriptive statistics:")
print(result_df[css_cols].describe().to_string())

print("\nMean by target:")
print(result_df.groupby("target")[css_cols].mean().to_string())

print("\nMedian by target:")
print(result_df.groupby("target")[css_cols].median().to_string())

Final dataset shape:
(3130, 49)

Missing values per feature:
css_class_token_entropy    0
css_reuse_factor           0
structured_naming_ratio    0
inline_style_density       0
css_var_count              0

Descriptive statistics:
       css_class_token_entropy  css_reuse_factor  structured_naming_ratio  inline_style_density  css_var_count
count              3130.000000       3130.000000              3130.000000           3130.000000    3130.000000
mean                  5.697901          5.987168                 0.776153              0.032816      20.759105
std                   0.835063          4.426011                 0.171124              0.046535      83.330268
min                   0.000000          0.000000                 0.000000              0.000000       0.000000
25%                   5.297944          3.601653                 0.667598              0.007776       0.000000
50%                   5.742655          5.090565                 0.822994              0.015930       0

In [8]:
# =========================
# 8. SAVE OUTPUT
# =========================
output_cols = [
    "url",
    "mbfc-credibility-rating",
    "target",
    "html_filename",
    "css_filename"
] + css_cols

final_css_df = result_df[output_cols].copy()
final_css_df.to_csv("css_regularity_features_3130.csv", index=False)

print("Saved: css_regularity_features_3130.csv")

Saved: css_regularity_features_3130.csv


In [20]:
import pandas as pd

test_df = pd.read_csv("css_regularity_features_3130.csv")
print(test_df.shape)
test_df.head()
test_df[test_df["target"] == 1].head(20)

(3130, 10)


,url,mbfc-credibility-rating,target,html_filename,css_filename,css_class_token_entropy,css_reuse_factor,structured_naming_ratio,inline_style_density,css_var_count
59,https://www.dailynk.com/,medium credibility,1,www.dailynk.com.html,www.dailynk.com.css,5.651016,3.717105,0.977876,0.037736,0
62,https://waitbutwhy.com/,medium credibility,1,waitbutwhy.com.html,waitbutwhy.com.css,6.000647,5.435780,0.742616,0.031037,0
107,https://www.thenews.mx/,medium credibility,1,www.thenews.mx.html,www.thenews.mx.css,6.011030,6.538462,0.802521,0.028139,3
124,https://www.digitalresearch.com/,medium credibility,1,www.digitalresearch.com.html,www.digitalresearch.com.css,5.453960,3.742647,0.976424,0.009772,110
135,https://www.patheos.com/,medium credibility,1,www.patheos.com.html,www.patheos.com.css,5.504913,6.325758,0.494611,0.005559,0
205,https://popculture.com/,medium credibility,1,popculture.com.html,popculture.com.css,5.773410,4.743405,0.929221,0.104418,0
224,http://www.2020insightllc.com/,medium credibility,1,www.2020insightllc.com.html,www.2020insightllc.com.css,5.047500,2.345455,0.550388,0.018315,0
252,https://www.wionews.com/,medium credibility,1,www.wionews.com.html,www.wionews.com.css,5.302168,7.285714,0.921053,0.045841,431
352,https://www.babnet.net/,medium credibility,1,www.babnet.net.html,www.babnet.net.css,5.000619,12.269737,0.582306,0.025452,0
414,http://www.newzjunky.com/,medium credibility,1,www.newzjunky.com.html,www.newzjunky.com.css,4.630657,12.002481,0.998760,0.012052,0


In [9]:
# =========================
# 9. RANDOM SAMPLE OF MEDIUM-CREDIBILITY CSS FILES
# =========================

# Medium credibility = target 1 in this notebook
medium_df = final_css_df[final_css_df["target"] == 1].copy()

print("Number of medium-credibility rows:")
print(medium_df.shape[0])

# Keep only rows whose CSS file actually exists in CSS_DIR
folder_css_files = {p.name for p in CSS_DIR.glob("*.css")}
medium_df = medium_df[medium_df["css_filename"].isin(folder_css_files)].copy()

print("\nNumber of medium-credibility rows with CSS file present:")
print(medium_df.shape[0])

# Random sample of 10
sample_medium_css = medium_df.sample(n=10, random_state=42).copy()

print("\nRandom sample of 10 medium-credibility CSS files:")
display(sample_medium_css[[
    "url",
    "mbfc-credibility-rating",
    "css_filename"
]].sort_values("css_filename"))

print("\nFilenames only:")
for fname in sample_medium_css["css_filename"].sort_values():
    print(fname)

Number of medium-credibility rows:
569

Number of medium-credibility rows with CSS file present:
569

Random sample of 10 medium-credibility CSS files:


,url,mbfc-credibility-rating,css_filename
2615,https://explainamerica.com/,medium credibility,explainamerica.com.css
919,https://indivisible.org/,medium credibility,indivisible.org.css
2684,https://tatumreport.com/,medium credibility,tatumreport.com.css
1020,https://www.boldprogressives.org/,medium credibility,www.boldprogressives.org.css
1043,https://www.gawker.com/,medium credibility,www.gawker.com.css
1172,https://www.huffpost.com/,medium credibility,www.huffpost.com.css
1929,https://www.kyivpost.com/,medium credibility,www.kyivpost.com.css
3005,https://www.mdpi.com/journal/behavsci,medium credibility,www.mdpi.com.css
3127,https://www.mdpi.com/journal/beverages,medium credibility,www.mdpi.com.css
2741,https://www.mdpi.com/journal/BDCC,medium credibility,www.mdpi.com.css



Filenames only:
explainamerica.com.css
indivisible.org.css
tatumreport.com.css
www.boldprogressives.org.css
www.gawker.com.css
www.huffpost.com.css
www.kyivpost.com.css
www.mdpi.com.css
www.mdpi.com.css
www.mdpi.com.css


In [10]:
# =========================
# 10. RANDOM SAMPLE OF HIGH-CREDIBILITY CSS FILES
# =========================

# High credibility = target 0 in this notebook
high_df = final_css_df[final_css_df["target"] == 0].copy()

print("Number of high-credibility rows:")
print(high_df.shape[0])

# Keep only rows whose CSS file actually exists in CSS_DIR
folder_css_files = {p.name for p in CSS_DIR.glob("*.css")}
high_df = high_df[high_df["css_filename"].isin(folder_css_files)].copy()

print("\nNumber of high-credibility rows with CSS file present:")
print(high_df.shape[0])

# Random sample of 10
sample_high_css = high_df.sample(n=10, random_state=42).copy()

print("\nRandom sample of 10 high-credibility CSS files:")
display(sample_high_css[[
    "url",
    "mbfc-credibility-rating",
    "css_filename"
]].sort_values("css_filename"))

print("\nFilenames only:")
for fname in sample_high_css["css_filename"].sort_values():
    print(fname)

Number of high-credibility rows:
2561

Number of high-credibility rows with CSS file present:
2561

Random sample of 10 high-credibility CSS files:


,url,mbfc-credibility-rating,css_filename
1527,https://bylinetimes.com/,high credibility,bylinetimes.com.css
1861,https://dcist.com/,high credibility,dcist.com.css
2430,https://japannews.yomiuri.co.jp/,high credibility,japannews.yomiuri.co.jp.css
1329,https://www.belfercenter.org/,high credibility,www.belfercenter.org.css
1367,https://www.caledoniacourier.com/,high credibility,www.caledoniacourier.com.css
1945,https://www.ctpublic.org/,high credibility,www.ctpublic.org.css
183,https://www.gapminder.org/,high credibility,www.gapminder.org.css
495,https://www.historynet.com/,high credibility,www.historynet.com.css
807,https://www.kens5.com/,high credibility,www.kens5.com.css
1639,https://www.nrc.nl/,high credibility,www.nrc.nl.css



Filenames only:
bylinetimes.com.css
dcist.com.css
japannews.yomiuri.co.jp.css
www.belfercenter.org.css
www.caledoniacourier.com.css
www.ctpublic.org.css
www.gapminder.org.css
www.historynet.com.css
www.kens5.com.css
www.nrc.nl.css
